In [8]:
# Basic libraries 
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pykalman import KalmanFilter
from itertools import combinations
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm
import pickle
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
sys.path.insert(0, PROJECT_ROOT)

In [9]:
# Scripts 
from scripts.src.cointegration import *
from scripts.src.kalman import *
from scripts.src.pairs import *
from scripts.src.trading_signal import *
from scripts.src.backtest import *
from scripts.src.plots import *
from scripts.src.utils import *

In [10]:
# Load in data from previous notebooks
df1_is = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df1_is")
df1_oos = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df1_oos")

df2_is = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df2_is_updated")
df2_oos = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/df2_oos_updated")

static_results_df = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/static_hedge_ratio")
dynamic_results_df = pd.read_csv("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dynamic_hedge_ratio")

In [11]:
# Load dictionaries from previous notebook
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/static_spreads.pkl", "rb") as f:
    static_spreads = pickle.load(f)
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/static_models.pkl", "rb") as f:
    static_models = pickle.load(f)

with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/dynamic_spreads.pkl", "rb") as f:
    dyamic_spreads = pickle.load(f)
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/dynamic_details.pkl", "rb") as f:
    dynamic_details = pickle.load(f)


# Implementation - Trading Signal
- For backtest here and compare the Kalman indicator to the baseline and a simple long strategy of only Asset A and Asset B (backtest.py)
- We implement a basic trading signal using standardised values for our dynamic spreads (z-scores)

The general rule is that:
- If the z-score > +2 - we short the spread 
- If the z-score < -2 - we long the spread
- If the z-score = 0 - we exit our position 

Mathematically, we can express it in a mapping:
$$
\text{Position}(z) =
\begin{cases}
-1 & \text{if } z > 2 \\
+1 & \text{if } z < -2 \\
\text{hold previous position} & \text{otherwise}
\end{cases}
$$

In [12]:
entry_values = [1.5, 2.0, 2.5]
exit_values = [0.0, 0.5, 1.0]

is_results = {}
summary_rows = []

for e in entry_values:
    for x in exit_values:
        key = f"entry_{e}_exit_{x}"

        signals_dict = generate_kalman_signals(
            dynamic_details,
            entry_z=e,
            exit_z=x
        )

        is_results[key] = signals_dict

        for (pair_name, R), df in signals_dict.items():
            summary_rows.append({
                "strategy": key,
                "pair": pair_name,
                "obs_cov": R,
                "entry_z": e,
                "exit_z": x,
                "num_trades": df["position"].diff().abs().sum() / 2,
                "mean_zscore": df["zscore"].mean(),
                "std_zscore": df["zscore"].std()
            })

is_trading_signals = pd.DataFrame(summary_rows)

In [13]:
is_trading_signals

,strategy,pair,obs_cov,entry_z,exit_z,num_trades,mean_zscore,std_zscore
0,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,0.5,1.5,0.0,26.0,-0.074323,1.254340
1,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,1.0,1.5,0.0,26.0,-0.075102,1.296052
2,entry_1.5_exit_0.0,700 HK Equity vs 1347 HK Equity,5.0,1.5,0.0,22.5,-0.075295,1.352083
3,entry_1.5_exit_0.0,1347 HK Equity vs 268 HK Equity,0.5,1.5,0.0,28.5,0.088018,1.284509
4,entry_1.5_exit_0.0,1347 HK Equity vs 268 HK Equity,1.0,1.5,0.0,27.5,0.090501,1.304701
...,...,...,...,...,...,...,...,...
157,entry_2.5_exit_1.0,3993 HK Equity vs 2689 HK Equity,1.0,2.5,1.0,18.5,0.079376,1.326103
158,entry_2.5_exit_1.0,3993 HK Equity vs 2689 HK Equity,5.0,2.5,1.0,18.0,0.066960,1.353544
159,entry_2.5_exit_1.0,1898 HK Equity vs 2386 HK Equity,0.5,2.5,1.0,20.0,-0.018652,1.295543
160,entry_2.5_exit_1.0,1898 HK Equity vs 2386 HK Equity,1.0,2.5,1.0,17.0,-0.025062,1.309947


In [14]:
# Save trading_signal details 
with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/is_trading_signals.pkl", "wb") as f:
    pickle.dump(is_trading_signals, f) 

with open("/Users/ivanhung/Documents/GitHub/applied-project-06039211/data/dictionaries/is_results.pkl", "wb") as f:
    pickle.dump(is_results, f)